In [0]:
from pyspark.sql.functions import col

gold_df = spark.table("stocks.gold.stocks_w_prev_returns").filter(
    col("date") >= '2025-01-01'
).na.drop()

In [0]:
%pip install xgboost

In [0]:
import xgboost as xgb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# --- Configuration ---
target_col = 'label'
id_cols = ['Date', 'company', 'feat_highest_return', 'feat_lowest_return', 'feat_daily_return', 'label']

feature_cols = [
    col for col in gold_df.columns
    if col not in [target_col] + id_cols
]

print("Fetching data to driver (Pandas)...")
pdf = gold_df.select(feature_cols + [target_col]).toPandas()

# --- CRITICAL FIX: Detect Class Count & Validate Labels ---
# XGBoost requires labels to be integers starting from 0 (e.g., 0, 1, 2)
num_classes = pdf[target_col].nunique()
print(f"Detected {num_classes} unique classes.")

# Verify labels are 0-indexed integers
# If your labels are strings or start at 1, uncomment the lines below to fix them:
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# pdf[target_col] = le.fit_transform(pdf[target_col])

# Determine Objective & Parameters based on class count
if num_classes > 2:
    objective_func = 'multi:softmax'
    xgb_kwargs = {'num_class': num_classes} # Explicitly pass num_class
else:
    objective_func = 'binary:logistic'
    xgb_kwargs = {} # No num_class needed for binary

# ---------------------------------------------------------

# 2. Split Data
X = pdf[feature_cols]
y = pdf[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_tune, X_val, y_tune, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

# 3. Tuning Loop
n_estimators_options = [200,250,300]
max_depth_options = [30,40,50]

best_accuracy = 0.0
best_params = {}

print(f"Starting Grid Search (Objective: {objective_func})...")

for n_est in n_estimators_options:
    for depth in max_depth_options:
        
        clf = xgb.XGBClassifier(
            n_estimators=n_est,
            max_depth=depth,
            learning_rate=0.1,
            objective=objective_func,
            tree_method='hist', 
            random_state=42,
            n_jobs=-1,
            **xgb_kwargs  # Unpacks {'num_class': N} if multi-class
        )
        
        clf.fit(X_tune, y_tune)
        preds = clf.predict(X_val)
        accuracy = accuracy_score(y_val, preds)
        
        print(f"   [Trees: {n_est}, Depth: {depth}] Accuracy: {accuracy:.4f}")
        
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_params = {'n_estimators': n_est, 'max_depth': depth}

print("-" * 30)
print(f"Best Accuracy: {best_accuracy:.4f}")

# 4. Final Training
print("Retraining final model...")
final_model = xgb.XGBClassifier(
    n_estimators=best_params['n_estimators'],
    max_depth=best_params['max_depth'],
    learning_rate=0.1,
    objective=objective_func,
    tree_method='hist',
    random_state=42,
    n_jobs=-1,
    **xgb_kwargs
)

final_model.fit(X_train, y_train)
print("Final model trained.")

In [0]:
from sklearn.metrics import roc_auc_score, average_precision_score

# 1. Generate probabilities (Probabilities are needed for ROC and PR)
# predict_proba returns [prob_class_0, prob_class_1] -> we take column [1]
probs = final_model.predict_proba(X_test)[:, 1]

# 2. Calculate Metrics
roc_auc = roc_auc_score(y_test, probs)
pr_auc = average_precision_score(y_test, probs)

print(f"areaUnderROC: {roc_auc}")
print(f"areaUnderPR: {pr_auc}")

In [0]:
import pandas as pd
import numpy as np

# --- Step 1: Recover Metadata ---
# Assuming 'df' is the dataframe you originally created X and y from
pdf = gold_df.toPandas() 
original_test_df = pdf.loc[X_test.index]

# --- Step 2: Get Predictions ---
probs = final_model.predict_proba(X_test)[:, 1]
preds = final_model.predict(X_test)

# --- Step 3: Combine Metadata + Features + Results ---
# We use the metadata as the base and add columns to it
results = original_test_df[['Date', 'company']].copy()

results['label'] = y_test
results['prediction'] = preds
results['prob_up'] = probs

# --- Step 4: Apply Logic ---
results['model_confidence'] = (results['prob_up'] - 0.5).abs() + 0.5
results['model_confidence'] = results['model_confidence'].round(4)

# --- Step 5: Sort and Display ---
sorted_df = results.sort_values(by="model_confidence", ascending=False)

sorted_df[["Date", "company", "label", "prediction", "prob_up", "model_confidence"]].display()

In [0]:
import pandas as pd
from sklearn.metrics import accuracy_score

# --- CONFIGURATION ---
CONFIDENCE_THRESHOLD = 0.7  # Only check trades where model is >60% sure
# ---------------------

# 1. Create a DataFrame with Labels and Predictions
# We start with the labels (Y_test) and align them
results = pd.DataFrame(index=X_test.index)
results['label'] = y_test

# Get raw probabilities (Class 1) and hard predictions
results['prob_up'] = final_model.predict_proba(X_test)[:, 1]
results['prediction'] = final_model.predict(X_test)

# Calculate confidence: Absolute distance from 0.5, normalized
results['model_confidence'] = (results['prob_up'] - 0.5).abs() + 0.5
results['model_confidence'] = results['model_confidence'].round(4)

# 2. Create the Subset
subset_df = results[results['model_confidence'] > CONFIDENCE_THRESHOLD]

# 3. Safety Check: Ensure we have data
total_rows = len(results)
subset_rows = len(subset_df)

print(f"Total Test Rows:      {total_rows}")
pct_retained = (subset_rows / total_rows) * 100 if total_rows > 0 else 0
print(f"High Confidence Rows: {subset_rows} ({pct_retained:.1f}% of data)")

# 4. Calculate Accuracy
if subset_rows > 0:
    # Use Scikit-Learn accuracy_score on the subset
    subset_acc = accuracy_score(subset_df['label'], subset_df['prediction'])
    
    print("-" * 30)
    print(f"Accuracy when >{CONFIDENCE_THRESHOLD*100:.0f}% sure:  {subset_acc:.2%}")
    print("-" * 30)

    # Optional: View the top winners
    # print(subset_df.sort_values("model_confidence", ascending=False).head())

else:
    print(f"⚠️ No trades met the {CONFIDENCE_THRESHOLD} threshold.")
    print("   Try lowering the threshold to 0.55 or 0.53.")

In [0]:
final_model

In [0]:
import mlflow
import mlflow.xgboost
from mlflow.models import infer_signature

# 1. Set the registry to Unity Catalog (UC)
# Replace 'main' with your catalog name and 'ml_schema' with your schema (database)
mlflow.set_registry_uri("databricks-uc") 
CATALOG = "stocks"
SCHEMA = "gold"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.xgb_direction_predictor"

# 2. Train and Log the model
# We use 'autolog' to automatically capture accuracy, AUC, and parameters shown in your screenshot.
mlflow.xgboost.autolog()

with mlflow.start_run() as run:
    # Assuming X_train / y_train are your training dataframes
    # If your model is already trained, you can skip .fit(), but re-running it here ensures metrics are captured.
    final_model.fit(X_train, y_train) 
    
    # 3. Register the model to Unity Catalog automatically
    # This creates a versioned model in the UC explorer
    mlflow.xgboost.log_model(
        xgb_model=final_model, 
        artifact_path="model",
        registered_model_name=MODEL_NAME,
        input_example=X_train.head(1) # Important for generating the schema signature
    )

print(f"Model registered to: {MODEL_NAME}")